# 03 — Baseline results

Reads `results/metrics/summary.parquet` produced by `experiments/run.py`
and renders comparison plots. To regenerate numbers:

```bash
for cfg in experiments/configs/baseline_*.yaml; do
  python -m experiments.run $cfg
done
```


In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import METRICS_DIR, save_figure
sns.set_theme(style='whitegrid', context='notebook')


In [ ]:
df = pd.read_parquet(METRICS_DIR / 'summary.parquet')
print('rows:', len(df), 'configs:', df['run_name'].nunique())
df.head()


In [ ]:
# F1 per model × dataset × metric, averaged over seeds.
agg = (df.groupby(['dataset', 'model', 'metric'])
         .agg(f1_mean=('f1', 'mean'), f1_std=('f1', 'std'), etapr_mean=('etapr_f1', 'mean'))
         .reset_index())
# For eTaPR rows, pick the etapr_f1 column into the f1 field.
agg.loc[agg['metric'] == 'etapr', 'f1_mean'] = agg.loc[agg['metric'] == 'etapr', 'etapr_mean']
agg[['dataset', 'model', 'metric', 'f1_mean', 'f1_std']]


In [ ]:
# Headline chart: F1 by metric, grouped by (dataset, model).
plot_df = agg.pivot_table(index=['dataset', 'model'], columns='metric', values='f1_mean')
plot_df = plot_df.reindex(columns=['pointwise', 'point_adjust', 'etapr'])
fig, ax = plt.subplots(figsize=(9, 5))
plot_df.plot(kind='bar', ax=ax, colormap='viridis')
ax.set_ylabel('F1')
ax.set_ylim(0, 1)
ax.set_title('Baseline F1 by evaluation metric')
ax.legend(title='metric')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
save_figure(fig, 'baseline_f1_by_metric', subdir='03_baseline')
plt.show()


## Markdown-ready table

In [ ]:
print(plot_df.round(3).to_markdown())
